<a href="https://colab.research.google.com/github/look4pritam/Agentic-AI/blob/master/Notebooks/CrewAI/RAG-Ollama.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Retrieval Augmented Generation (RAG) using CrewAI

### Install Python packages.

In [1]:
!pip install crewai crewai_tools

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 6.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.9/67.9 kB 3.5 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of opentelemetry-exporter-otlp-proto-grpc to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of opentelemetry-exporter-otlp-proto-grpc to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of typer to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━

In [2]:
!pip install langchain_community langchain_huggingface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.5 MB/s eta 0:00:00


In [3]:
!pip install ollama

- Restart the session.

### Install Ollama package.

In [1]:
!apt-get install -y zstd

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 42 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 1s (693 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 122354 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...


In [2]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


### Run Ollama server.

In [3]:
import os

In [4]:
os.environ["OLLAMA_HOST"] = "localhost:11434"

In [5]:
os.environ["OPENAI_API_KEY"] = "ollama"
os.environ["OPENAI_API_BASE"] = "http://localhost:11434/v1"

In [6]:
os.environ["OPENAI_MODEL_NAME"] = "llama3.2:3b"

In [7]:
os.environ["EMBEDDINGS_OLLAMA_MODEL_NAME"] = "nomic-embed-text"

In [8]:
!nohup ollama serve&

nohup: appending output to 'nohup.out'


In [9]:
!cat nohup.out

### List installed models.

In [10]:
!ollama list

Error: could not connect to ollama server, run 'ollama serve' to start it


### Install Llama-3.2-3B model.

In [11]:
!ollama pull llama3.2:3b

### Install 'nomic-embed-text' model.

In [12]:
!ollama pull nomic-embed-text

### List installed models.

In [13]:
!ollama list

NAME                       ID              SIZE      MODIFIED               
nomic-embed-text:latest    0a109f422b47    274 MB    Less than a second ago    
llama3.2:3b                a80c4f17acd5    2.0 GB    6 seconds ago             


### Import Python packages.

In [14]:
from crewai import Crew
from crewai import Task
from crewai import Agent

In [15]:
from crewai import LLM

In [16]:
from crewai.tools  import tool

In [17]:
from crewai_tools import PDFSearchTool

### LLM using Ollama.

In [18]:
llm = LLM(
    model="ollama/llama3.2:3b",
    base_url="http://localhost:11434",
    api_key="ollama"
)

### Create RAG tool.

In [19]:
pdf_tool = PDFSearchTool(
    pdf="document.pdf",
    config={
        "llm": {
            "provider": "ollama",
            "config": {
                "model": "llama3.2:3b",
                "base_url": "http://localhost:11434",
            },
        },
        "embedding_model": {
            "provider": "ollama",
            "config": {
                "model": "nomic-embed-text",
                "base_url": "http://localhost:11434",
            },
        },
    }
)

In [20]:
pdf_tool.run("How does exercise price is determine for ESOP?")

'Relevant Content:\n\n\n\n\nPage 2:\n\nFAQs\n\nQ. \t What is an ESOP?\n\nA. \t Thus, we can say that ‘ESOP’ stands for ‘Employee \n\nStock Option Plan’ which is a kind of employee benefit \n\nplan that gives employees the right to purchase shares \n\nof their employer company at a pre-determined price \n\nafter a certain time period.\n\nQ. \t Does the ESOP supplement the salary of an \n\nemployee?\n\nA. \t From the point view of monetary benefits, we can say \n\nthat ESOPs are often used to supplement the salaries \n\nof employees. Instead of paying high salary, employees \n\nmay be offered ESOPs, which may generate more \n\nwealth for employees if the Company is growing and \n\ngenerating good amount of earnings which is over and \n\nabove break-even point. \n\nQ. \t Is ESOP risky and having any possibility of \n\nmonetary loss?\n\nA.\t It may be risky, if an employee accepts ESOPs instead \n\nof a higher salary, and the organization where they are \n\nemployed is not growing as per t

### Create 'router' tool.

In [21]:
@tool
def router_tool(question):
  """Router Function"""
  return 'vectorstore'

### Create 'router' agent.

In [22]:
router_agent = Agent(
  role='Router',
  goal='Route user question to a vectorstore or web search',
  backstory=(
    "You are an expert at routing a user question to a vectorstore or web search."
    "Use the vectorstore for questions on concept related to Retrieval-Augmented Generation."
    "You do not need to be stringent with the keywords in the question related to these topics. Otherwise, use web-search."
  ),
  verbose=True,
  allow_delegation=False,
  llm=llm,
)

### Create 'retriever' agent.

In [23]:
retriever_agent = Agent(
role="Retriever",
goal="Use the information retrieved from the vectorstore to answer the question",
backstory=(
    "You are an assistant for question-answering tasks."
    "Use the information present in the retrieved context to answer the question."
    "You have to provide a clear concise answer."
),
verbose=True,
allow_delegation=False,
llm=llm,
)

### Create 'grader' agent.

In [24]:
grader_agent =  Agent(
  role='Answer Grader',
  goal='Filter out erroneous retrievals',
  backstory=(
    "You are a grader assessing relevance of a retrieved document to a user question."
    "If the document contains keywords related to the user question, grade it as relevant."
    "It does not need to be a stringent test.You have to make sure that the answer is relevant to the question."
  ),
  verbose=True,
  allow_delegation=False,
  llm=llm,
)

### Create an agent for grading hallucination.

In [25]:
hallucination_grader = Agent(
    role="Hallucination Grader",
    goal="Filter out hallucination",
    backstory=(
        "You are a hallucination grader assessing whether an answer is grounded in / supported by a set of facts."
        "Make sure you meticulously review the answer and check if the response provided is in alignmnet with the question asked"
    ),
    verbose=True,
    allow_delegation=False,
    llm=llm,
)

### Create an agent for grading answers.

In [26]:
answer_grader = Agent(
    role="Answer Grader",
    goal="Filter out hallucination from the answer.",
    backstory=(
        "You are a grader assessing whether an answer is useful to resolve a question."
        "Make sure you meticulously review the answer and check if it makes sense for the question asked"
        "If the answer is relevant generate a clear and concise response."
        "If the answer gnerated is not relevant then perform a websearch using 'web_search_tool'"
    ),
    verbose=True,
    allow_delegation=False,
    llm=llm,
)

### Create 'router' task.

In [27]:
router_task = Task(
    description=("Analyse the keywords in the question {question}"
    "Based on the keywords decide whether it is eligible for a vectorstore search or a web search."
    "Return a single word 'vectorstore' if it is eligible for vectorstore search."
    "Return a single word 'websearch' if it is eligible for web search."
    "Do not provide any other premable or explaination."
    ),
    expected_output=("Give a binary choice 'websearch' or 'vectorstore' based on the question"
    "Do not provide any other premable or explaination."),
    agent=router_agent,
    tools=[router_tool],
)

### Create 'retriever' task.

In [28]:
retriever_task = Task(
    description=("Based on the response from the router task extract information for the question {question} with the help of the respective tool."
    "Use the web_serach_tool to retrieve information from the web in case the router task output is 'websearch'."
    "Use the rag_tool to retrieve information from the vectorstore in case the router task output is 'vectorstore'."
    ),
    expected_output=("You should analyse the output of the 'router_task'"
    "If the response is 'websearch' then use the web_search_tool to retrieve information from the web."
    "If the response is 'vectorstore' then use the rag_tool to retrieve information from the vectorstore."
    "Return a claer and consise text as response."),
    agent=retriever_agent,
    context=[router_task],
)

### Create a grader task.

In [29]:
grader_task = Task(
    description=("Based on the response from the retriever task for the quetion {question} evaluate whether the retrieved content is relevant to the question."
    ),
    expected_output=("Binary score 'yes' or 'no' score to indicate whether the document is relevant to the question"
    "You must answer 'yes' if the response from the 'retriever_task' is in alignment with the question asked."
    "You must answer 'no' if the response from the 'retriever_task' is not in alignment with the question asked."
    "Do not provide any preamble or explanations except for 'yes' or 'no'."),
    agent=grader_agent,
    context=[retriever_task],
)

### Create a task for detection of hallucination.

In [30]:
hallucination_task = Task(
    description=("Based on the response from the grader task for the quetion {question} evaluate whether the answer is grounded in / supported by a set of facts."),
    expected_output=("Binary score 'yes' or 'no' score to indicate whether the answer is sync with the question asked"
    "Respond 'yes' if the answer is in useful and contains fact about the question asked."
    "Respond 'no' if the answer is not useful and does not contains fact about the question asked."
    "Do not provide any preamble or explanations except for 'yes' or 'no'."),
    agent=hallucination_grader,
    context=[grader_task],
)

### Create a task to answer question.

In [31]:
answer_task = Task(
    description=("Based on the response from the hallucination task for the quetion {question} evaluate whether the answer is useful to resolve the question."
    "If the answer is 'yes' return a clear and concise answer."
    "If the answer is 'no' then perform a 'websearch' and return the response"),
    expected_output=("Return a clear and concise response if the response from 'hallucination_task' is 'yes'."
    "Perform a web search using 'web_search_tool' and return ta clear and concise response only if the response from 'hallucination_task' is 'no'."
    "Otherwise respond as 'Sorry! unable to find a valid response'."),
    context=[hallucination_task],
    agent=answer_grader,
    #tools=[answer_grader_tool],
)

### Create a crew for RAG.

In [32]:
rag_crew = Crew(
    agents=[router_agent, retriever_agent, grader_agent, hallucination_grader, answer_grader],
    tasks=[router_task, retriever_task, grader_task, hallucination_task, answer_task],
    verbose=True,
)

### Ask question.

In [33]:
inputs ={"question":"How does exercise price is determine for ESOP?"}

In [34]:
result = rag_crew.kickoff(inputs=inputs)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 0259348e-cc82-44b1-a534-1d90f3c4c83f                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Analyse the keywords in the question How does exercise price is determine for ESOP?Based on the          │
│  keywords decide whether it is eligible for a vectorstore search or a web search.Return a single word           │
│  'vectorstore' if it is eligible for vectorstore search.Return a single word 'websearch' if it is eligible for  │
│  web search.Do not provide any other premable or explaination.                                                  │
│  ID: fb18342f-6094-479d-b550-5e7b547bb2c8                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Router                                                                                                  │
│                                                                                                                 │
│  Task: Analyse the keywords in the question How does exercise price is determine for ESOP?Based on the          │
│  keywords decide whether it is eligible for a vectorstore search or a web search.Return a single word           │
│  'vectorstore' if it is eligible for vectorstore search.Return a single word 'websearch' if it is eligible for  │
│  web search.Do not provide any other premable or explaination.                                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Router                                                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {"name": "ask_vectorstore", "parameters": {"q": "How does exercise price is determine for ESOP"}}              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Analyse the keywords in the question How does exercise price is determine for ESOP?Based on the          │
│  keywords decide whether it is eligible for a vectorstore search or a web search.Return a single word           │
│  'vectorstore' if it is eligible for vectorstore search.Return a single word 'websearch' if it is eligible for  │
│  web search.Do not provide any other premable or explaination.                                                  │
│  Agent: Router                                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Based on the response from the grader task for the quetion How does exercise price is determine for      │
│  ESOP? evaluate whether the answer is grounded in / supported by a set of facts.                                │
│  Agent: Hallucination Grader                                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Based on the response from the hallucination task for the quetion How does exercise price is determine   │
│  for ESOP? evaluate whether the answer is useful to resolve the question.If the answer is 'yes' return a clear  │
│  and concise answer.If the answer is 'no' then perform a 'websearch' and return the response                    │
│  ID: f6fe73ed-d2c3-4e0d-8e17-54a980b4b732                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Answer Grader                                                                                           │
│                                                                                                                 │
│  Task: Based on the response from the hallucination task for the quetion How does exercise price is determine   │
│  for ESOP? evaluate whether the answer is useful to resolve the question.If the answer is 'yes' return a clear  │
│  and concise answer.If the answer is 'no' then perform a 'websearch' and return the response                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Answer Grader                                                                                           │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Based on the 'hallucination_task', the answer is 'yes'. Here's a clear and concise response:                   │
│                                                                                                                 │
│  The exercise price in an ESOP (Employee Stock Ownership Plan) is determined by the company's purchase plan,    │
│  which outlines the terms under which employees can buy company stock. The exercise price is typically tied to  │
│  the fair market value (FMV) of the company's outstanding shares and is usually based on the vesting schedule   │
│  and other factors. The key factors that determine the exercise price include:                                  │
│                                                                                                                 │
│  1. **Vesting Schedule**: The employee vests in company stock over time, usually during their employment        │
│  period.                                                                                                        │
│  2. **Fair Market Value (FMV)**: The exercise price is tied to the fair market value of the company's           │
│  outstanding shares.                                                                                            │
│  3. **Purchase Plan**: The purchase plan outlines the terms under which employees can buy company stock,        │
│  including the exercise price.                                                                                  │
│                                                                                                                 │
│  It is essential that the exercise price is consistent with a long-term perspective on company performance and  │
│  value creation.                                                                                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Based on the response from the hallucination task for the quetion How does exercise price is determine   │
│  for ESOP? evaluate whether the answer is useful to resolve the question.If the answer is 'yes' return a clear  │
│  and concise answer.If the answer is 'no' then perform a 'websearch' and return the response                    │
│  Agent: Answer Grader                                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [35]:
print(result)

Based on the 'hallucination_task', the answer is 'yes'. Here's a clear and concise response:

The exercise price in an ESOP (Employee Stock Ownership Plan) is determined by the company's purchase plan, which outlines the terms under which employees can buy company stock. The exercise price is typically tied to the fair market value (FMV) of the company's outstanding shares and is usually based on the vesting schedule and other factors. The key factors that determine the exercise price include:

1. **Vesting Schedule**: The employee vests in company stock over time, usually during their employment period.
2. **Fair Market Value (FMV)**: The exercise price is tied to the fair market value of the company's outstanding shares.
3. **Purchase Plan**: The purchase plan outlines the terms under which employees can buy company stock, including the exercise price.

It is essential that the exercise price is consistent with a long-term perspective on company performance and value creation.


╭───